# Tutorial on CNN architectures interpretability and optimization

In [ ]:
## Install some of the necessary packages in google colab
# !pip install livelossplot
#!pip install optuna
#!pip install GPUtil
#!pip install tabulate
## !pip install umap-learn
#!pip install shap

## 1) Import packages

In [ ]:
import os
import sys
from sys import stdout
import logging
import pickle

import random
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import matplotlib.lines as mlines
import matplotlib.colors as colors
from matplotlib.collections import LineCollection
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib import cm
import seaborn as sns 
import pandas as pd
from IPython.display import clear_output, Image, display

# import scipy.io as sio
from scipy.signal import savgol_filter
from scipy import signal, stats 
import tqdm
# from itertools import permutations
# import sklearn
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cross_decomposition import PLSRegression
# from sklearn.decomposition import PCA
# from sklearn.manifold import TSNE
# from umap import UMAP
from sklearn.model_selection import train_test_split, cross_val_score , KFold
from sklearn.metrics import root_mean_squared_error, mean_squared_error, r2_score 
from sklearn.utils import shuffle

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.activations import elu
from tensorflow.keras import layers, Model
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv1D, Reshape, Dense, Flatten, Lambda
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, Callback,  ModelCheckpoint
from tensorflow.keras.utils import to_categorical, plot_model 
# import tensorflow_addons as tfa


## Use liveslossplot for training visualization in real time
from livelossplot import PlotLossesKerasTF
import optuna
import GPUtil
from tabulate import tabulate
import psutil
import platform
import socket
from datetime import datetime 

# import umap
# import shap


For future refence, we list the main versions of the main software packages and used hardware

In [ ]:
## Choose just GPU:1
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## Print machine info
print('\n--------  Running @',socket.gethostname(),' using ', platform.platform(),'--------\n' )
# Get current date and time
now = datetime.now()
dt_string = now.strftime("%d/%m/%Y %H:%M:%S")
print("Last run at =", dt_string) 

## Print versions and hardware info
print('\n-------- SOFTWARE INFO --------')
print('Python ', sys.version)
print('Tensorflow ', tf.__version__)
# print('Tensorflow add-ons ', tfa.__version__)
print('tqdm ', tqdm.__version__)
print('Numpy ', np.__version__)
print('Pandas', pd.__version__)
print('Optuna ', optuna.__version__)
# print('Scikit-learn ', sklearn.__version__)
# print('livelossplot ', livelossplot.__version__)

## print hardware info
print('\n-------- HARDWARE INFO --------')
# CPU
print('CPU:', platform.processor())
print("\tPhysical cores:", psutil.cpu_count(logical=False))
print("\tTotal cores:", psutil.cpu_count(logical=True))
cpufreq = psutil.cpu_freq()
print(f"\tMax Frequency: {cpufreq.max:.2f}Mhz")
# RAM
print(f'RAM: {int(np.round(psutil.virtual_memory().total / (1024. **3)))} Gb')
# GPU
print('GPU available: ', tf.config.list_physical_devices('GPU'))
print("="*40, "GPU Details", "="*40)
gpus = GPUtil.getGPUs()
list_gpus = []
for gpu in gpus:
    # get the GPU id
    gpu_id = gpu.id
    # name of GPU
    gpu_name = gpu.name
    # get % percentage of GPU usage of that GPU
    gpu_load = f"{gpu.load*100}%"
    # get free memory in MB format
    gpu_free_memory = f"{gpu.memoryFree}MB"
    # get used memory
    gpu_used_memory = f"{gpu.memoryUsed}MB"
    # get total memory
    gpu_total_memory = f"{gpu.memoryTotal}MB"
    # get GPU temperature in Celsius
    gpu_temperature = f"{gpu.temperature} °C"
    gpu_uuid = gpu.uuid
    list_gpus.append((
        gpu_id, gpu_name, gpu_load, gpu_free_memory, gpu_used_memory,
        gpu_total_memory, gpu_temperature, gpu_uuid
    ))

print(tabulate(list_gpus, headers=("id", "name", "load", "free memory", "used memory", "total memory",
                                   "temperature", "uuid")))

print('\nIs CUDA accessible by the GPU? ', tf.test.is_built_with_cuda())

## 2) Help functions
In this section we implement a series of help functions that will be used during the optimization procedure. Run every cell once to ensure that all help functions are loaded.

In [ ]:
## Define random seeds ir order to maintain reproducible results through multiple testing phases
def reproducible_comp():
    os.environ['PYTHONHASHSEED'] = '0'
    np.random.seed(42)
    random.seed(42)
    tf.random.set_seed(42)

reproducible_comp()

A functions to compute grad-cam scores.

In [ ]:
## Adapted for 1d input data from https://keras.io/examples/vision/grad_cam/
    
def make_gradcam_heatmap(input_data, model, last_conv_layer_name, pred_index=None):
    # Forward Pass: The grad_model is a sub-model of your original model which provides outputs of both 
    # the final model output and the output of the convolution layer. The forward pass is done in the 
    # line last_conv_layer_output, preds = grad_model(input_adata). Here, input_data would be the 
    # input spectrum.
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(input_data)
        # if pred_index is None:
        #     pred_index = tf.argmax(preds[0])
        output_channel = preds[:, 0]

    # Compute Gradients: The gradient computation is performed in the line 
    # grads = tape.gradient(output_channel, last_conv_layer_output). This computes the gradients of the 
    # final output (output_channel) with respect to the outputs of the convolution layer (last_conv_layer_output).
    ## This is the gradient of the output neuron (top predicted or chosen)
    ## with regard to the output feature map of the last conv layer
    grads = tape.gradient(output_channel, last_conv_layer_output)

    # Weighted Combination of Feature Maps: First, you compute the pooled gradients in the line 
    # pooled_grads = tf.reduce_mean(grads, axis=(1)), which represent the mean importance of each feature map. 
    # Then, these are used to create a weighted combination of the feature maps of the last convolutional layer. 
    # The line heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis] is where this happens. 
    # The output of this operation is a weighted combination of the feature maps, and it is of the same 
    # shape as the output of the convolution layer.
    ## This is a vector where each entry is the mean intensity of the gradient
    ## over a specific feature map channel
    pooled_grads = tf.reduce_mean(grads, axis=(1))

    # Average Pooling: The operation tf.squeeze(heatmap) then reduces the dimensions of the heatmap tensor 
    # to a 1D tensor (a heatmap) of the same shape as the input spectrum.
    ## We multiply each channel in the feature map array
    ## by "how important this channel is" with regard to the top predicted class
    ## then sum all the channels to obtain the heatmap class activation
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # For visualization purpose, we will also normalize the heatmap between 0 & 1
    # or if this line is commented, we should normalize the heatmap in the plotting function
    #   heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()    

## Code adapted from: https://github.com/dpsanders/matplotlib-examples/blob/master/colorline.ipynb

## Defines a function colorline that draws a (multi-)colored 2D line with coordinates x and y.
## The color is taken from optional data in z, and creates a LineCollection.

## z can be:
## - empty, in which case a default coloring will be used based on the position along the input arrays
## - a single number, for a uniform color [this can also be accomplished with the usual plt.plot]
## - an array of the length of at least the same length as x, to color according to this data
## - an array of a smaller length, in which case the colors are repeated along the curve

## The function colorline returns the LineCollection created, which can be modified afterwards.


## Data manipulation:

def make_segments(x, y):
    '''
    Create list of line segments from x and y coordinates, in the correct format for LineCollection:
    an array of the form   numlines x (points per line) x 2 (x and y) array
    '''

    points = np.array([x, y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    
    return segments


## Interface to LineCollection:

def colorline(x, y, z=None, cmap=plt.get_cmap('jet'), norm=colors.Normalize(vmin=0.0, vmax=1), linewidth=3, alpha=1.0):
    '''
    Plot a colored line with coordinates x and y
    Optionally specify colors in the array z
    Optionally specify a colormap, a norm function and a line width
    '''
    
    # Default colors equally spaced on [0,1]:
    if z is None:
        z = np.linspace(0.0, 1, len(x))
           
    # Special case if a single number:
    if not hasattr(z, "__iter__"):  # to check for numerical input -- this is a hack
        z = np.array([z])
        
    z = np.asarray(z)
    
    segments = make_segments(x, y)
    lc = LineCollection(segments, array=z, cmap=cmap, norm=norm, linewidth=linewidth, alpha=alpha)
    
    ax = plt.gca()
    ax.add_collection(lc)
#     plt.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax)
    return lc
        
    
def clear_frame(ax=None): 
    # Taken from a post by Tony S Yu
    if ax is None: 
        ax = plt.gca() 
    ax.xaxis.set_visible(False) 
    ax.yaxis.set_visible(False) 
    for spine in ax.spines.itervalues(): 
        spine.set_visible(False) 

Axiliary function to compute error metrics and make prediction plots

In [ ]:
## Function to compute metrics and make prediction plots using train and test data
def plot_prediction2(Y_train, Y_test, Y_train_pred, Y_test_pred, title, savefig=False, figname=None):

    ## Compute train error scores
    score_p0 = r2_score(Y_train, Y_train_pred)
    mse_p0 = mean_squared_error(Y_train, Y_train_pred)
    rmse_p0 = np.sqrt(mse_p0)

    ## Compute test error scores
    score_p2 = r2_score(Y_test, Y_test_pred)
    mse_p2 = mean_squared_error(Y_test, Y_test_pred)
    rmse_p2 = np.sqrt(mse_p2)

    print('ERROR METRICS: \t TRAIN  \t\t TEST')
    print('------------------------------------------------------')
    print('R2:   \t\t %5.3f  \t\t %5.3f'  % (score_p0, score_p2 ))
    print('RMSE: \t\t %5.3f  \t\t %5.3f' % (rmse_p0, rmse_p2))

    #### Plot regression for model predicted data
    ## Get plot limits
    Y = np.concatenate([y_train, y_test])

    rangey = np.max(Y) - np.min(Y)
    rangex = np.max(Y) - np.min(Y)
    ## x=y line and +- 1std upper and lower bowndaries
    xy_x=np.ravel([np.min(Y)-0.1*rangex, np.max(Y)+0.1*rangex])
    xy_y=np.ravel([np.min(Y)-0.1*rangey, np.max(Y)+0.1*rangey])

    plt.figure(figsize=(5,5))
    z = np.polyfit(np.ravel(Y_test), np.ravel(Y_test_pred), 1)
    print('Fit result: Y=',z[1], ' + ', z[0],' * X')
    ax = plt.subplot(aspect=1)
    ax.plot(xy_x, xy_y, 'k--', linewidth=2, label=None)
    ax.scatter(Y_train, Y_train_pred, c='gray', marker='o', s=20, alpha=0.66, label='train')
    ax.scatter(Y_test,Y_test_pred, s=40, marker='o', facecolors='None', edgecolors='r', label='test')
    # Calculate the range of x-axis based on Y_train and Y_test
    x_min = min(np.min(Y_train), np.min(Y_test))
    x_max = max(np.max(Y_train), np.max(Y_test))
    # Create an array spanning the range of x-axis
    x_range = np.linspace(x_min, x_max, num=100)
    ax.plot(x_range, z[1]+z[0]*x_range, c='blue', linewidth=2,label='linear fit')
    plt.xlim(xy_x)
    plt.ylim(xy_y)
    # ax.plot(x_range, x_range, 'k--', linewidth=1.5, label='y=x')
    plt.ylabel('Predicted')
    plt.xlabel('Measured')
    plt.title(title)
    plt.legend(loc=4, frameon=False)

    # Print the scores on the plot
    plt.text(np.min(xy_x)+0.05*rangex, np.max(xy_y)-0.1*rangey, 'R$^{2}=$ %5.2f'  % score_p2, fontsize=13)
    plt.text(np.min(xy_x)+0.05*rangex, np.max(xy_y)-0.15*rangey, 'RMSE: %5.2f' % rmse_p2, fontsize=13)
    if savefig==True:
        plt.savefig(figname, dpi=150)
        print('Figure saved')
    else:
        plt.show()
    return


## Function to compute metrics and make prediction plots (custom version of previous function)
def plot_Y_prediction(Y, Y_pred, title, ax, savefig=False, figname=None):
    ## Compute train error scores
    score_p0 = r2_score(Y, Y_pred)
    mse_p0 = mean_squared_error(Y, Y_pred)
    rmse_p0 = np.sqrt(mse_p0)
#     print('R2: \t\t %5.3f '  % (score_p0))
#     print('RMSE: \t\t %5.3f' % (rmse_p0))
    ## Plot regression for PLS predicted data
    rangey = max(Y) - min(Y)
    rangex = max(Y_pred) - min(Y_pred)
    fig=plt.figure(figsize=(3,3))
    z = np.polyfit(np.ravel(Y), np.ravel(Y_pred), 1)
    print('Fit result: Y=',z[1], ' + ', z[0],' * X')
    # ax = plt.subplot(aspect=1)
    ax.scatter(Y,Y_pred,c='k',marker='o',s=10, alpha=0.6)
    ax.plot(Y, z[1]+z[0]*Y, c='blue', linewidth=2,label='linear fit')
    ax.plot(Y, Y, 'k--', linewidth=1.5, label='y=x')
    ax.set_ylabel('Predicted')
    ax.set_xlabel('Measured')
    ax.set_title(title)
#     plt.legend(loc=4)
    # Print the scores on the plot
    ax.text(min(Y_pred)+0.02*rangex, max(Y)-0.1*rangey, 'R$^{2}=$ %5.3f'  % score_p0)
    ax.text(min(Y_pred)+0.02*rangex, max(Y)-0.15*rangey, 'RMSE: %5.3f' % rmse_p0)
    if savefig==True:
        plt.savefig(figname, dpi=150)
        print('Figure saved')
    else:
        plt.show()
    return




### Functions for computing and optimizing PLS models ############

# from chemometrics_analysis_help import *

def error_metrics(y_true0, y_pred0):
    y_true=np.ravel(y_true0)
    y_pred=np.ravel(y_pred0)
    ## R squared R2 (based on the Pearson correlation) 
    R2 = stats.pearsonr(y_true.squeeze(),y_pred.squeeze())[0]**2
    ## Root Mean Squared Error (RMSE)
    RMSE = np.sqrt(mean_squared_error(y_true, y_pred))
    ## Prediction Gain (PG)
     # initialize PG vector with the mean value
    PG0 = np.zeros(len(y_pred)) + np.mean(y_true)
     # Now we compute the rms error between this preciction (mean value) and the validation set
    PG0_MSE= np.sqrt(mean_squared_error(y_true, PG0))
    PG= PG0_MSE / RMSE    
    ## Coefficient of Variation (CVAR) 
    CVAR = np.round(100.*RMSE/np.mean(y_true),2)
    SDR = np.std(y_true) / RMSE
    return np.round(R2,3), np.round(RMSE,3), np.round(PG,3), np.round(CVAR,3), np.round(SDR,3)


def pls_optimization_cv_stop2(x_train, y_train, nmax=20, Nfolds=5, plot_opt=False, stop_criteria=0.01):
    """
    This function computes the optimal number of LVs for a PLS model using cross-validation using as stop criteria
    the LV that produces a gain in the CV RMSE lower than 1% of the previous LV.
    """
    ## List to store the CV RMSE for each LV
    cv_rmse=[]

    print('\nComputing optimal number of LVs for PLS model in the range 1 to {}...\n'.format(nmax))
    component = np.arange(1, nmax+1)
    previous_cv_rmse = None
    bestLV_stop = None

    print('Stop criteria: {}% gain in RMSE'.format(stop_criteria*100))

    for i in component:
        pls = PLSRegression(n_components=i, scale=True)
        cv_score=cross_val_score(pls, x_train, y_train, cv=KFold(Nfolds, shuffle = True, random_state=42),\
                        n_jobs=-1, scoring='neg_mean_squared_error', error_score=0)
        current_cv_rmse = np.round(np.sqrt(-np.mean(cv_score)),3)
        ## Check if the current CV RMSE is less than 1% of the previous CV RMSE
        if (previous_cv_rmse is not None) and (current_cv_rmse <= np.min(cv_rmse)):
            percent_diff = abs((current_cv_rmse - previous_cv_rmse) / previous_cv_rmse)
            # print(f'Compute difference percentage between current LV={i} and previous LV={i-1} CV RMSE -> {np.round(percent_diff*100,3)}%')
            if percent_diff <= stop_criteria and bestLV_stop is None:
                print(f"Stopping criteria reached, {np.round(percent_diff*100,3)}%. Saving component number.")
                ## The previous LV is the last one that adds more than 1% of gain in RMSE
                bestLV_stop = i-1
                ## Save the RMSE of the previous LV
                RMSE_stop = previous_cv_rmse
        previous_cv_rmse = current_cv_rmse
        cv_rmse.append(current_cv_rmse)
        criterion_flag = '1% gain'

    ## if the 1% criteria returns no LV, then the bestLV_stop is the one where the RMSE is minimum
    if bestLV_stop is None:
        print(f'Stop criteria of {stop_criteria*100}% gain in RMSE not reached. Using minimum RMSE.')
        bestLV_stop = np.argmin(cv_rmse)+1
        RMSE_stop = np.min(cv_rmse)
        criterion_flag = 'minimum RMSE'

    RMSE_best = np.round(np.min(RMSE_stop),3)
    print(f'Suggested number of LV based on {Nfolds}-fold CV RMSE using {criterion_flag}: {bestLV_stop}')
    print(f'{Nfolds} CV RMSE: {RMSE_best}')
    stdout.write("\n")
    if plot_opt is True:
        plt.figure(figsize=(9,3))
        ax1=plt.subplot()
        ax1.plot(component[:len(cv_rmse)], np.array(cv_rmse), '-v', color = 'blue', mfc='blue')
        if bestLV_stop is not None:
            ax1.plot(component[bestLV_stop-1], [RMSE_best], 'P', ms=10, mfc='red',label='LV chosen')
        plt.xlabel('Number of PLS components')
        plt.ylabel('Mean of '+str(Nfolds)+'CV RMSE ')
        ax1.axvline(x=bestLV_stop, color='red', lw=1,linestyle='--')
        ax1.set_xticks(component)
        plt.xlim(0, nmax+1)
        # plt.title('# PLS components')
        plt.legend()
        plt.grid(alpha=0.33)
        plt.show()
    return bestLV_stop, RMSE_best




def pls_prediction_metrics2(l, x_train, y_train, x_test, y_test, xname, yname, lv, verbose=True, plot_pred=False, plot_vip=False):
    """
    USE: pls_prediction_metrics(x_train, y_train, x_test, y_test, yname, lv, plot_pred=False)
    Data X and Y should be numpy arrays...
    #################
    l: vector with wavelenghts for plots x scale
    x_train, y_train: train dataset
    x_test, y_test:   test dataset
    xname:  name of x data (for plots)
    yname:  name of y data (for plots)
    lv:     number of latent variables for PLS model
    verbose: True (False) for output text with results
    plot_pred: False (True) for plot predictions
    plot_vip: False (True) for plot of VIP scores
    ##################
    OUTPUT: several error metrics for train and test sets...
    """
    ## Define PLS with suggested optimal number of components and fit train data
    pls1 = PLSRegression(n_components=lv, scale=True)

    ## Fit PLS model to train data
    pls1.fit(x_train, y_train)

    ## Get predictions for train and test sets
    y_train_pred = pls1.predict(x_train)
    y_test_pred = pls1.predict(x_test)

    ## Compute error metrics
    R2_train, RMSE_train, PG_train, CVAR_train, SDR_train = error_metrics(y_train, y_train_pred)
    R2_test, RMSE_test, PG_test, CVAR_test, SDR_test = error_metrics(y_test, y_test_pred)

    if verbose == True:
        print('\nError metrics for best PLS model with LV = {}'.format(lv))
        print('METRIC \t TRAIN \t TEST ')
        print('R2     \t {:0.3f}\t {:0.3f}'.format(R2_train,R2_test))
        print('RMSE   \t {:0.3f}\t {:0.3f}'.format(RMSE_train,RMSE_test))
        # print('PG   \t {:0.3f}\t {:0.3f}'.format(PG_train,PG_test))
        print('CVAR   \t {:0.3f}\t {:0.3f}'.format(CVAR_train,CVAR_test))
        print('SDR  \t {:0.3f}\t {:0.3f}'.format(SDR_train,SDR_test))

    ## Plots: MSE vs. PLS LV and regression for best PLS model
    # Get plot limits
    Y = np.concatenate([y_train, y_test])

    rangey = np.max(Y) - np.min(Y)
    rangex = np.max(Y) - np.min(Y)

    # x=y line and +- 1std upper and lower bowndaries
    xy_x=np.ravel([np.min(Y)-0.1*rangex, np.max(Y)+0.1*rangex])
    xy_y=np.ravel([np.min(Y)-0.1*rangey, np.max(Y)+0.1*rangey])

    xy_y_up=xy_y+np.std(Y)
    xy_y_down=xy_y-np.std(Y)

    if plot_pred is True:
        ## linear fit to predicted test data
        plt.figure(figsize=(5,5))
        # plt.title(yname+' prediction using '+xname+' data')

        ## fit the test data
        z = np.polyfit(np.ravel(y_test), np.ravel(y_test_pred), 1)
        print('Fit result: Y=',z[1], ' + ', z[0],' * X')
        ax = plt.subplot()
        ax.plot(xy_x, xy_y, 'k--', linewidth=2, label = None)
        # plt.fill_between(xy_x, xy_y_down, xy_y_up, alpha=0.2)
        ax.scatter(y_train,y_train_pred,c='gray',s=26, marker='o', alpha=0.66, label='Train')
        ax.scatter(y_test,y_test_pred,s=40, marker='o', facecolors='None', edgecolors='r', label='Test')
        x_range = np.linspace(np.min(Y), np.max(Y), num=100)
        ax.plot(x_range, z[1]+z[0]*x_range, c='blue', linewidth=3, label = None)
        plt.xlim(xy_x)
        plt.ylim(xy_y)
        plt.ylabel('Predicted '+yname, fontsize=10)
        plt.xlabel('Measured '+yname, fontsize=10)
        plt.legend(loc=4)
        # Print the test error metrics on the plot
        plt.text(np.min(xy_x)+0.05*rangex, np.max(xy_y)-0.1*rangey, 'R$^{2}=$ %5.2f'  % R2_test, fontsize=13)
        plt.text(np.min(xy_x)+0.05*rangex, np.max(xy_y)-0.15*rangey, 'RMSE: %5.2f' % RMSE_test, fontsize=13)
        # plt.text(np.min(xy_x)+0.05*rangex, np.max(xy_y)-0.2*rangey, 'PG: %5.2f' % PG_test, fontsize=13)
        # plt.text(np.min(xy_x)+0.05*rangex, np.max(xy_y)-0.25*rangey, 'CVar: %5.2f%%' % CVAR_test, fontsize=13)
        plt.show()

    if plot_vip is True:
        pls_vip=vip(pls1)
        fig, ax = plt.subplots(figsize=(12,3))
        # plt.title('PLS VIP scores ')
        plt.ylabel('VIP score', fontsize=14)
        plt.xlabel('Wavelength (nm)', fontsize=14)
        ax.plot(l,pls_vip,'k',label='VIP scores')
        ax.set_ylim(np.min(pls_vip), np.max(pls_vip))
        plt.axhline(1,color='k', linestyle='--',linewidth=0.75)
        plt.legend()
        plt.show()

    return R2_train, RMSE_train, PG_train, CVAR_train, SDR_train, R2_test, RMSE_test, PG_test, CVAR_test, SDR_test, y_train_pred, y_test_pred


def pls_explained_variance(pls, X, Y_true, do_plot=True):
    r2 = np.zeros(pls.n_components)
    x_transformed = pls.transform(X) # Project X into low dimensional basis
    for i in range(0, pls.n_components):
        Y_pred = (np.dot(x_transformed[:, i][:, np.newaxis],
                         pls.y_loadings_[:, i][:, np.newaxis].T) * pls._y_std
                  + pls._y_mean)
        r2[i] = r2_score(Y_true, Y_pred)
        overall_r2 = r2_score(Y_true, pls.predict(X))  # Use all components together.

    if do_plot:
        plt.figure(figsize=(5,5))
        component = np.arange(pls.n_components) + 1
        plt.bar(component, r2)
        plt.xticks(component)
        plt.xlabel('Number of PLS components')
        plt.ylabel('Explained variance')
        # plt.title(f'Summed individual r2: {np.sum(r2):.3f}, '
        #           f'Overall r2: {overall_r2:.3f}')
        plt.show()

    return r2, overall_r2


def vip(model):
    """
    Compute the VIP scores of a trained PLS model
    """
    t = model.x_scores_
    w = model.x_weights_
    q = model.y_loadings_
    p, h = w.shape
    vips = np.zeros((p,))
    s = np.diag(t.T @ t @ q.T @ q).reshape(h, -1)
    total_s = np.sum(s)
    for i in range(p):
        weight = np.array([ (w[i,j] / np.linalg.norm(w[:,j]))**2 for j in range(h) ])
        vips[i] = np.sqrt(p*(s.T @ weight)/total_s)
    return vips


### Functions used in the original multifruit HPO study to define the CNN architectures

In [ ]:
## Define the model
def create_model_1(num_FC_layers, num_FC_units, filter_size, DROPOUT, reg_beta):
    ## Layers dimensions
    INPUT_DIMS = np.shape(x_train_scaled)[1]
    CONV1D_DIMS = INPUT_DIMS
    K_NUMBER = 1
    K_WIDTH = filter_size
    K_STRIDE = 1
    OUTPUT_DIMS = 1
    
    
    ## Global (all layers) L2 regularizer parameter
    beta = reg_beta
    K_REG = tf.keras.regularizers.l2(beta)
    
    ## For the sake of simplicity we do the weights initialization for multiple layers here
    K_INIT = tf.keras.initializers.he_normal(seed=123)
    
    ## Architecture of the main model
    ## This way of implementing the model is analogous to the way we previously did although it is done
    ## in an alternative way that allows a bit more coding freedom. 
    model_cnn = keras.Sequential(name='MODEL_CNN_V1')
    model_cnn.add(keras.layers.Reshape((INPUT_DIMS, 1),input_shape=(INPUT_DIMS,), name='reshape'))
    model_cnn.add(keras.layers.Conv1D(filters=K_NUMBER, \
                                      kernel_size=K_WIDTH, \
                                      strides=K_STRIDE, \
                                      padding='same', \
                                      kernel_initializer=K_INIT,\
                                      kernel_regularizer=K_REG,\
                                      activation='elu',\
                                      input_shape=(CONV1D_DIMS,1), name='CONVOLUTIONAL'))
    
    model_cnn.add(keras.layers.Flatten(name='FLATTEN'))
    
    ## For the FC layer block, we implement a loop that adds dense layers with a certain number of units
    ## followed by a dropout layer (with a certain dropout rate)
    ## The number of layers, units, dropout rate, etc. will be optmized. Note that a dropout rate = 0 is
    ## the same as excluding that dropout layer... 
    for i in range(0, num_FC_layers):
        model_cnn.add(keras.layers.Dense(num_FC_units[i], \
                                         kernel_initializer=K_INIT, \
                                         kernel_regularizer=K_REG,\
                                         activation='elu', name='DENSE_'+str(i)))
        if i != num_FC_layers - 1:  # Only add dropout if it's not the last iteration
            model_cnn.add(keras.layers.Dropout(DROPOUT[i], name='DROPOUT_'+str(i)))  

    ## Final layer for multi-label classification
    model_cnn.add(keras.layers.Dense(OUTPUT_DIMS, kernel_initializer=K_INIT, \
                                        activation='linear', name='OUTPUT'))
                             
    
    return model_cnn



def run_Nx_cnnR_v1_metrics(N, NUM_FC_LAYERS, NUM_FC_UNITS, FILTER_SIZE, DROPOUT, REG_BETA, BATCH_SIZE, LR,
                            EPOCHS, XTRAIN, YTRAIN, XTEST, YTEST, MODEL_NAME):
    ''' 
    ########### Compute the metrics for 10x CNN models  ################
    1st:Determine the best number of epochs for training the model
        1) Shuffle train data
        2) Create a 5-fold CV scheme where the train data is split into cal and val sets
        3) Train model in 5-fold CV monitoring the val loss in early stopping
           During training, the rdlr callback monitors the val_loss
        4) Save the epoch where early stopping was triggered into a list
    2nd: Determine the best number of epochs for training the model based on the mean of the epochs where early stopping was triggered
    3rd: Train the model 10x (on randomized versions of the train set) with the best number of epochs
         No validation split or validation data is used and the 'rdlr' callback monitors the training loss
    4th: Compute error metrics on train and test sets
    5th: Compute the mean and std of the error metrics over the 10x models
    '''
    
    ## Create lists to store metrics  
    RMSE_train_list = []
    R2_train_list = []
    RMSE_test_list = []
    R2_test_list = []
    epochs_list = []

    ## Callbacks
    progressbar=tfa.callbacks.TQDMProgressBar(show_epoch_progress=False, update_per_second=5)
    early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', min_delta=5e-4, patience=52, mode='auto', 
                                               restore_best_weights=True, verbose=1)  
    rdlr_cv = keras.callbacks.ReduceLROnPlateau(patience=25, factor=0.5, min_lr=1e-6, monitor='val_loss', verbose=0)
    
    ## Shuffle train data to mix samples with the same seed used in the HPO
    x_train_scaled_shuf, y_train_shuf = shuffle(XTRAIN, YTRAIN, random_state=42)
    

    if EPOCHS == 'auto':
        ########### Train the model 5 times in CV to find the optimal number of epochs ###########
        # create KFold object
        kf = KFold(n_splits = 5)

        print('Train data shuffled... Determining optimal number of epochs using 5-fold CV...\n')

        ## Loop for training the model 5 times under different calibration/validation splits
        for i, (cal_index, val_index) in enumerate(kf.split(x_train_scaled_shuf)):

            ## Define the cal and val sets for this iteration
            x_cal_scaled_shuf, x_val_scaled_shuf = x_train_scaled_shuf[cal_index], x_train_scaled_shuf[val_index]
            y_cal_shuf, y_val_shuf = y_train_shuf[cal_index], y_train_shuf[val_index] 

            ## Create a new model instance
            MODEL = create_model_1(NUM_FC_LAYERS, NUM_FC_UNITS, FILTER_SIZE, DROPOUT, REG_BETA)
            ## Compile the model defining the optimizer, the loss function and the metrics to track during training
            MODEL.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR), loss="mse", metrics=["mse"]) 

            ## CONTROL
            # print('Pre train filter weights: ', np.ravel(MODEL.get_weights()[0])[:5])

            ## Train the model for a max 500 of epochs on the cal set and track the val loss
            MODEL.fit(x_cal_scaled_shuf, y_cal_shuf, batch_size = BATCH_SIZE, epochs = 500,\
                      validation_data = (x_val_scaled_shuf, y_val_shuf),\
                      callbacks=[rdlr_cv, early_stop],\
                      verbose=0)

            ## CONTROL
            # print('Post train filter weights: ', np.ravel(MODEL.get_weights()[0])[:5])

            ## Store the number of epochs the model was trained for. If early stopping was not triggered, store 500
            if early_stop.stopped_epoch==0:
                print('\nReached the max training epochs')
                epochs_list.append(500)
            else:
                epochs_list.append(early_stop.stopped_epoch)

            keras.backend.clear_session()
        ## END CROSS-VALIDATION LOOP FOR EPOCHS DETERMINATION

        ## Get the mean of the training epochs identified in 10k cross validation
        max_epoch = np.mean(epochs_list)
        print('\nThe model trained for', int(max_epoch), 'epochs on average in 5-fold cross-validation.')
    else:
        max_epoch = EPOCHS
        print('\nThe model will be trained for', int(max_epoch), 'epochs set manually')
    
    print('\n\n---------------------------------------------------------------------------------------------------------\n')
    print('Training final model for', int(max_epoch), 'epochs on the full train set ',N,' times')

    ## Callbacks redefinition so that the rdlr monitors just the train loss
    rdlr = keras.callbacks.ReduceLROnPlateau(patience=25, factor=0.5, min_lr=1e-6, monitor='loss', verbose=0)

    y_train_shuf_list = []
    y_test_pred_list = []

    ## Train 10 models for this number of epochs, and compute mean error of predictions
    for i in np.arange(0,N,1):
    
        ## Create a new model instance (reset weights)
        MODEL = create_model_1(NUM_FC_LAYERS, NUM_FC_UNITS, FILTER_SIZE, DROPOUT, REG_BETA)
        ## Compile the model defining the optimizer, the loss function and the metrics to track during training
        MODEL.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR), loss="mse", metrics=["mse"])  

        ## Save the best model based on the cal loss (the val loss is not used at any point during training)
        checkpointer = ModelCheckpoint(filepath=MODEL_NAME, monitor='loss', verbose=0, save_best_only=True)

        ## Shuffle training data to ensure different batches in each training run (the random_state is set for reproducibility)
        x_train_scaled_shuf, y_train_shuf = shuffle(XTRAIN, YTRAIN, random_state = int(i))
        ## Store the shuffled train data for later use
        y_train_shuf_list.append(y_train_shuf)

        print(f'\n Run {i}...\n Train data reshuffled...\n First 3 train samples: {np.ravel(y_train_shuf[0:4])}')

        ## CONTROL
        # print('Pre train filter weights: ', np.ravel(MODEL.get_weights()[0])[:5])

        ## train the model each time on a different shuffle of the training data
        MODEL.fit(x_train_scaled_shuf, y_train_shuf, batch_size = BATCH_SIZE, shuffle=False, epochs = int(max_epoch), 
                       callbacks=[rdlr, checkpointer, progressbar], verbose=0)
        
        ## CONTROL
        # print('Post train filter weights: ', np.ravel(MODEL.get_weights()[0])[:5])
    
        print(f'\n Training completed... \n Loading best model weights from {MODEL_NAME}... \n Computing metrics for Run {i}...')
        ## Load the best model weights
        MODEL.load_weights(MODEL_NAME)

        ## Compute RMSE metrics for TRAIN and TEST sets
        y_train_pred = MODEL.predict(x_train_scaled_shuf)
        y_test_pred = MODEL.predict(XTEST)
        ## Save the test predictions into a list
        y_test_pred_list.append(y_test_pred)

        ## Compute train error scores 
        scoreR2_train = r2_score(y_train_shuf, y_train_pred)
        rmse_train = np.sqrt(mean_squared_error(y_train_shuf, y_train_pred))
        RMSE_train_list.append(rmse_train)
        R2_train_list.append(scoreR2_train)

        ## Compute test error scores 
        scoreR2_test = r2_score(YTEST, y_test_pred)
        rmse_test = np.sqrt(mean_squared_error(YTEST, y_test_pred))
        RMSE_test_list.append(rmse_test)
        R2_test_list.append(scoreR2_test)

                
        print('\nEVAL '+str(i)+' ERROR METRICS: \t TRAIN  \t\t TEST')
        print('\t R2: \t\t %5.3f  \t\t %5.3f'  % (scoreR2_train, scoreR2_test ))
        print('\t RMSE: \t\t %5.3f \t\t\t %5.3f' % (rmse_train, rmse_test))
                
        ## Clear clutter from previous session
        keras.backend.clear_session()
        print('\n Keras backend cleared...')

        ## Plot last run predictions
        # if i==9:
        #    plot_prediction2(y_train_shuf,  YTEST, y_train_pred,  y_test_pred, title='CNN v1',\
        #                     savefig=False, figname='cnn_v1_prediction_run.png')
    ### END FOR CYLE

    ## Compute mean variabilioty of 10 runs
    RMSE_train_mean = np.mean(RMSE_train_list)
    RMSE_test_mean = np.mean(RMSE_test_list)
    R2_train_mean = np.mean(R2_train_list)
    R2_test_mean = np.mean(R2_test_list)
    RMSE_train_std = np.std(RMSE_train_list)
    RMSE_test_std = np.std(RMSE_test_list)

    ## Compute y_test_pred mean and compute ensemble R2 and RMSE
    y_test_pred_ensemble = np.mean(y_test_pred_list, axis=0)
    R2_test_ensemble = r2_score(np.ravel(YTEST), np.ravel(y_test_pred_ensemble))   
    RMSE_test_ensemble = np.sqrt(mean_squared_error(np.ravel(YTEST), np.ravel(y_test_pred_ensemble)))

    print('\n------------------------------------------------------')
    print(f'MEAN ERROR METRICS: \t TRAIN  \t TEST \t\t {N} ENSEMBLE TEST')
    print('------------------------------------------------------')
    print('R2: \t\t %5.3f  \t\t %5.3f \t\t %5.3f'  % (R2_train_mean, R2_test_mean, R2_test_ensemble ))
    print('RMSE: \t\t %5.3f+-%3.3f \t\t %5.3f+-%3.3f \t %3.5f'  % (RMSE_train_mean, RMSE_train_std, RMSE_test_mean, RMSE_test_std, RMSE_test_ensemble))
    print('------------------------------------------------------')
    return






def create_model_2(num_FC_layers, num_FC_units, filter_size, DROPOUT, reg_beta):
    ## Layers dimensions
    INPUT_DIMS = np.shape(x_train_scaled)[1]
    K_NUMBER = 1
    K_WIDTH = filter_size
    K_STRIDE = 1
    REG_OUTPUT_DIMS = 1
    
    ## Global (all layers) L2 regularizer parameter
    beta = reg_beta
    K_REG = tf.keras.regularizers.l2(beta)
    
    ## Weights initialization for multiple layers
    K_INIT = tf.keras.initializers.he_normal(seed=123)
    
    ## Architecture of the main model
    input_layer = layers.Input(shape=(INPUT_DIMS,), name='INPUT')
    x = layers.Reshape((INPUT_DIMS, 1),name='RESHAPE')(input_layer)
    x = layers.Conv1D(filters=K_NUMBER, 
                      kernel_size=K_WIDTH, 
                      strides=K_STRIDE, 
                      padding='same', 
                      kernel_initializer=K_INIT,
                      kernel_regularizer=K_REG,
                      activation='elu',
                      name='CONVOLUTIONAL')(x)
    
    x = layers.Flatten(name='FLATTEN')(x)

    for i in range(0, num_FC_layers):
        x = layers.Dense(num_FC_units[i], 
                         kernel_initializer=K_INIT, 
                         kernel_regularizer=K_REG,
                         activation='elu', 
                         name='DENSE'+str(i))(x)
        if i != num_FC_layers - 1:  # Only add dropout if it's not the last iteration
            x = layers.Dropout(DROPOUT[i], name='DROPOUT'+str(i))(x)
    
    # Regression output
    reg_output = layers.Dense(REG_OUTPUT_DIMS, 
                              kernel_initializer=K_INIT, 
                              activation='linear', 
                              name='REG_OUTPUT')(x)

    # Create the model with multiple outputs
    model_cnn = Model(inputs=input_layer, outputs=[reg_output], name='MODEL_CNN_V2')
    
    return model_cnn




def run_Nx_cnnRC_v2_metrics(N, NUM_FC_LAYERS, NUM_FC_UNITS, FILTER_SIZE, DROPOUT, REG_BETA, BATCH_SIZE, LR,
                            EPOCHS, XTRAIN, YTRAIN, LABELS_TRAIN, CLASS_TRAIN,
                            XTEST, YTEST, CLASS_TEST, MODEL_NAME):
    ''' 
    ########### Compute the metrics for 10x CNN models  ################
    1st:Determine the best number of epochs for training the model
        1) Shuffle train data
        2) Create a 5-fold CV scheme where the train data is split into cal and val sets
        3) Train model in 5-fold CV monitoring the val loss in early stopping
           During training, the rdlr callback monitors the val_loss
        4) Save the epoch where early stopping was triggered into a list
    2nd: Determine the best number of epochs for training the model based on the mean of the epochs where early stopping was triggered
    3rd: Train the model 10x (on randomized versions of the train set) with the best number of epochs
         No validation split or validation data is used and the 'rdlr' callback monitors the training loss
    4th: Compute error metrics on train and test sets
    5th: Compute the mean and std of the error metrics over the 10x models
    '''
   
    
    ## Create lists to store metrics  
    RMSE_train_list = []
    R2_train_list = []
    RMSE_test_list = []
    R2_test_list = []
    ACC_train_list = []
    ACC_test_list = []
    epochs_list = []

    ## Callbacks
    progressbar=tfa.callbacks.TQDMProgressBar(show_epoch_progress=False, update_per_second=5)
    early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', min_delta=5e-4, patience=52, mode='auto', 
                                               restore_best_weights=True, verbose=1)  
    rdlr_cv = keras.callbacks.ReduceLROnPlateau(patience=25, factor=0.5, min_lr=1e-6, monitor='val_loss', verbose=0)
    
    ## Shuffle train data to mix samples with the same seed used in the HPO
    x_train_scaled_shuf, y_train_shuf, labels_train_shuf = shuffle(XTRAIN, YTRAIN, LABELS_TRAIN, random_state=42)
    
    if EPOCHS == 'auto':
        ########### Train the model 5 times in CV to find the optimal number of epochs ###########
        # create KFold object
        kf = KFold(n_splits = 5)

        print('Train data shuffled... Determining optimal number of epochs using 5-fold CV...\n')

        ## Loop for training the model 5 times under different calibration/validation splits
        for i, (cal_index, val_index) in enumerate(kf.split(x_train_scaled_shuf)):

            ## Define the cal and val sets for this iteration
            x_cal_scaled_shuf, x_val_scaled_shuf = x_train_scaled_shuf[cal_index], x_train_scaled_shuf[val_index]
            y_cal_shuf, y_val_shuf = y_train_shuf[cal_index], y_train_shuf[val_index]
            labels_cal_shuf, labels_val_shuf = labels_train_shuf[cal_index], labels_train_shuf[val_index]

            ## Create a new model instance
            MODEL = create_model_2(NUM_FC_LAYERS, NUM_FC_UNITS, FILTER_SIZE, DROPOUT, REG_BETA)
            ## Compile the model defining the optimizer, the loss function and the metrics to track during training
            MODEL.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR), loss=['categorical_crossentropy', 'mse'],
                          metrics=[['acc'], ['mse']]) 

            ## Train the model for a max 500 of epochs on the cal set and track the val loss
            MODEL.fit(x_cal_scaled_shuf, [labels_cal_shuf, y_cal_shuf], batch_size = BATCH_SIZE, epochs = 500,\
                      validation_data = (x_val_scaled_shuf, [labels_val_shuf, y_val_shuf]),\
                      callbacks=[rdlr_cv, early_stop],\
                      verbose=0)

            ## Store the number of epochs the model was trained for. If early stopping was not triggered, store 500
            if early_stop.stopped_epoch==0:
                print('\nReached the max training epochs')
                epochs_list.append(500)
            else:
                epochs_list.append(early_stop.stopped_epoch)

            keras.backend.clear_session()
        ## END CROSS-VALIDATION LOOP FOR EPOCHS DETERMINATION

        ## Get the mean of the training epochs identified in 10k cross validation
        max_epoch = np.mean(epochs_list)
        print('\nThe model trained for', int(max_epoch), 'epochs on average in 5-fold cross-validation.')
    else:
        max_epoch = EPOCHS
        print('\nThe model will be trained for', int(max_epoch), 'epochs set manually')   

    print('\n\n---------------------------------------------------------------------------------------------------------\n')
    print('Training final model for', int(max_epoch), 'epochs on the full train set ', N,' times')

    ## Callbacks redefinition so that the rdlr monitors just the train loss
    rdlr = keras.callbacks.ReduceLROnPlateau(patience=25, factor=0.5, min_lr=1e-6, monitor='loss', verbose=0)

    y_train_shuf_list = []
    y_test_pred_list = []
    class_train_shuf_list = []
    labels_train_shuf_list = []
    labels_test_pred_list = []
    
    ## Train 10 models for this number of epochs, and compute mean error of predictions
    for i in np.arange(0,N,1):
    
        ## Create a new model instance (reset weights)
        MODEL = create_model_2(NUM_FC_LAYERS, NUM_FC_UNITS, FILTER_SIZE, DROPOUT, REG_BETA)
        ## Compile the model defining the optimizer, the loss function and the metrics to track during training
        MODEL.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR), loss=['categorical_crossentropy', 'mse'],
                      metrics=[['acc'], ['mse']]) 

        ## Save the best model based on the cal loss (the val loss is not used at any point during training)
        checkpointer = ModelCheckpoint(filepath=MODEL_NAME, monitor='loss', verbose=0, save_best_only=True)

        ## Shuffle training data to ensure different batches in each training run (the random_state is set for reproducibility)
        x_train_scaled_shuf, y_train_shuf, labels_train_shuf, class_train_shuf = shuffle(XTRAIN, YTRAIN,
                                                                                         LABELS_TRAIN, CLASS_TRAIN,
                                                                                         random_state = int(i))
        ## Store the shuffled train data for later use
        y_train_shuf_list.append(y_train_shuf)
        class_train_shuf_list.append(class_train_shuf)
        labels_train_shuf_list.append(labels_train_shuf)

        print(f'\n Run {i}...\n Train data reshuffled...\n First 3 train samples: {np.ravel(y_train_shuf[0:4])}')

        ## train the model each time on a different shuffle of the training data
        MODEL.fit(x_train_scaled_shuf, [labels_train_shuf, y_train_shuf], 
                       batch_size = BATCH_SIZE, shuffle=False, epochs = int(max_epoch), 
                       callbacks=[rdlr, checkpointer, progressbar], verbose=0)
        
         
        print(f'\n Training completed... \n Loading best model weights from {MODEL_NAME}... \n Computing metrics for Run {i}...')
        ## Load the best model weights
        MODEL.load_weights(MODEL_NAME)

        ## Compute RMSE metrics for TRAIN and TEST sets
        y_train_pred = MODEL.predict(x_train_scaled_shuf)[1]
        y_test_pred = MODEL.predict(XTEST)[1]
        ## Save the test predictions into a list
        y_test_pred_list.append(y_test_pred)

        ## Compute train error scores 
        scoreR2_train = r2_score(y_train_shuf, y_train_pred)
        rmse_train = np.sqrt(mean_squared_error(y_train_shuf, y_train_pred))
        RMSE_train_list.append(rmse_train)
        R2_train_list.append(scoreR2_train)

        ## Compute test error scores 
        scoreR2_test = r2_score(YTEST, y_test_pred)
        rmse_test = np.sqrt(mean_squared_error(YTEST, y_test_pred))
        RMSE_test_list.append(rmse_test)
        R2_test_list.append(scoreR2_test)

        ## Compute metrics Train and Test classifications metrics
        label_train_pred = MODEL.predict(x_train_scaled_shuf, verbose=0)[0]
        label_test_pred  = MODEL.predict(XTEST, verbose=0)[0]
        ## store the predicted labels for later use
        labels_test_pred_list.append(label_test_pred)

        ## Convert the one-hot encoded labels into a single integer class
        class_train_pred = np.argmax(label_train_pred, axis = 1)
        class_test_pred = np.argmax(label_test_pred, axis = 1)
        
        acc_train = accuracy_score(class_train_shuf, class_train_pred)
        acc_test = accuracy_score(CLASS_TEST, class_test_pred)
        
        ## append values to list
        ACC_train_list.append(acc_train)
        ACC_test_list.append(acc_test)
        
        print('\nEVAL '+str(i)+' ERROR METRICS: \t TRAIN  \t\t TEST')
        print('\t R2: \t\t %5.3f  \t\t %5.3f'  % (scoreR2_train, scoreR2_test ))
        print('\t RMSE: \t\t %5.3f \t\t\t %5.3f' % (rmse_train, rmse_test))
        print('\t ACC: \t\t %5.3f \t\t\t %5.3f' % (acc_train, acc_test))
        
        ## Clear clutter from previous session
        keras.backend.clear_session()
        print('\n Keras backend cleared...')

        ## Plot last run predictions
        # if i==9:
        #     plot_prediction2(y_train_shuf,  YTEST, y_train_pred,  y_test_pred, title='CNN v2',\
        #                      savefig=False, figname='cnn_v2_prediction_run.png')
    ### END FOR CYLE

    ## Compute mean of 10 runs
    RMSE_train_mean = np.mean(RMSE_train_list)
    RMSE_test_mean = np.mean(RMSE_test_list)
    ACC_train_mean = np.mean(ACC_train_list)
    ACC_test_mean = np.mean(ACC_test_list)
    R2_train_mean = np.mean(R2_train_list)
    R2_test_mean = np.mean(R2_test_list)
    RMSE_train_std = np.std(RMSE_train_list)
    RMSE_test_std = np.std(RMSE_test_list)

    ## Compute y_test_pred mean and compute ensemble R2 and RMSE
    y_test_pred_ensemble = np.mean(y_test_pred_list, axis=0)
    R2_test_ensemble = r2_score(np.ravel(YTEST), np.ravel(y_test_pred_ensemble))   
    RMSE_test_ensemble = np.sqrt(mean_squared_error(np.ravel(YTEST), np.ravel(y_test_pred_ensemble)))
    ## Since the labels are probabilities, we can find the mean probability (ensemble) and compute its class and accuracy
    labels_test_pred_ensemble = np.mean(labels_test_pred_list, axis=0)
    ACC_test_ensemble = accuracy_score(np.ravel(CLASS_TEST), np.ravel(np.argmax(labels_test_pred_ensemble, axis = 1)))

    print('\n------------------------------------------------------')
    print(f'MEAN ERROR METRICS: \t TRAIN  \t TEST \t\t {N} ENSEMBLE TEST')
    print('------------------------------------------------------')
    print('R2: \t\t %5.3f  \t\t %5.3f \t\t %5.3f'  % (R2_train_mean, R2_test_mean, R2_test_ensemble ))
    print('RMSE: \t\t %5.3f+-%3.3f \t\t %5.3f+-%3.3f \t %3.5f'  % (RMSE_train_mean, RMSE_train_std, RMSE_test_mean, RMSE_test_std, RMSE_test_ensemble))
    print('ACC: \t\t\t %5.3f \t\t\t %5.3f \t\t %5.3f' % (ACC_train_mean, ACC_test_mean, ACC_test_ensemble))
    print('------------------------------------------------------')
    return




Set parameters for graphics formating

In [ ]:
## Graphics settings
## Setting the font sizes for comming figures
plt.style.use("default")
SMALL_SIZE = 10
MEDIUM_SIZE = 12
BIGGER_SIZE = 14

## uncomment for Latex graphics formating
# plt.rcParams.update({
#     "text.usetex": True,
#     "font.family": "serif",
#     "font.sans-serif": ["Times"]})

# plt.rc('text', usetex=True)
plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=SMALL_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

## 3) Data wrangling

### 3.1) Loading the "CEOT 2010 pear dataset"

Log(1/R) from pears (CEOT dataset from 2010), measured with an Hamamatsu TG-9405CA spectrometer. 

This data set is a subset of the one used in https://doi.org/10.3390/s19235165, with trimmed sample space and spectral range.

In [ ]:
## Import the train and test data
data_train = pd.read_csv('data/data_train_small.csv')
data_test = pd.read_csv('data/data_test_small.csv')

## Define the spectra...
x_train = data_train.iloc[:, :-3]
x_test = data_test.iloc[:, :-3]
## ... and the brix values
y_train = data_train['brix']
y_test = data_test['brix']
## ... and the other labels
temp_train = data_train['temperature']
temp_test = data_test['temperature']
size_train = data_train['calib']
size_test = data_test['calib']

## wavelength for plots
# convert w list to float array
w = np.array([float(i) for i in data_train.columns.values[:-3]])

In [ ]:
plt.figure(figsize=(12,3))
plt.subplot(1, 2, 1)
plt.title('Pear spectra (train)')
plt.plot(w, x_train.T)
plt.ylabel('Log(1/R) (a.u)')
plt.xlabel('Wavelength (nm)')
plt.subplot(1, 2, 2)
plt.title('Pear spectra (test)')
plt.plot(w, x_test.T)
plt.ylabel('Log(1/R) (a.u)')
plt.xlabel('Wavelength (nm)')
plt.show()

In [ ]:
plt.figure(figsize=(12,3))
plt.subplot(1, 2, 1)
plt.plot(y_train, 'ro', markersize=3, label='Train')
plt.plot(np.arange(len(y_train), len(y_train)+len(y_test)), y_test, 'bo', markersize=3,label='Test')
plt.title('Brix values')
plt.ylabel('Brix')
plt.xlabel('Sample number')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(temp_train, 'ro', markersize=3, label='Train')
plt.title('Temperature values')
plt.ylabel('Temperature (°C)')
plt.xlabel('Sample number')
plt.plot(np.arange(len(temp_train), len(temp_train)+len(temp_test)), temp_test, 'bo', markersize=3, label='Test')
plt.show()

## 4) PLS modelling

A previous preprocessing study found that 1d preprocessing is the one that work best for PLS on this dataset.

In [ ]:
## Compute 1st Sav-Gol derivative
x_train_1d=savgol_filter(x_train,51,2,deriv=1)
x_test_1d=savgol_filter(x_test,51,2,deriv=1)

plt.figure(figsize=(12,3))
plt.subplot(1, 2, 1)
plt.title('1st Savitzky-Golay derivative (train)')
plt.plot(w, x_train_1d.T)
plt.ylabel('1d(Log(1/R))')
plt.xlabel('Wavelength (nm)')
plt.subplot(1, 2, 2)
plt.title('1st Savitzky-Golay derivative (test)')
plt.plot(w, x_test_1d.T)
plt.ylabel('1d(Log(1/R))')
plt.xlabel('Wavelength (nm)')
plt.tight_layout()
plt.show()

In [ ]:
best_LV, CV_RMSE = pls_optimization_cv_stop2(x_train_1d, y_train, nmax=20, plot_opt=True, stop_criteria=0.005)
print('Chosen number of LV:', best_LV, '\nCV RMSE:', CV_RMSE)

f = pls_prediction_metrics2(w, x_train_1d, y_train, x_test_1d, y_test, 
                            '1st deriv','Brix', lv=best_LV, 
                            plot_pred=True, plot_vip=False)

In [ ]:
pls2 = PLSRegression(n_components=8)
pls2.fit(x_train_1d, y_train)
pls_explained_variance(pls2, x_train_1d, y_train)

## 5) CNN models

In [ ]:
## Since the test set is unknown (we are not suppose to have access to it during the
## optimization of the model) the scaling process should take this into account. We
## have to define a scaler based only on the train data, and apply it to the test data.

def standardize_column(X_train, X_test):
    ## We train the scaler on the full train set and apply it to the test set
    scaler = StandardScaler().fit(X_train)
    ## for columns we fit the scaler to the train set and apply it to the test set
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return [X_train_scaled, X_test_scaled]

In [ ]:
## smoothing 
# x_train_smooth = savgol_filter(x_train, 25, 2)
# x_test_smooth = savgol_filter(x_test, 25, 2)
## 2d
# x_train_2d=savgol_filter(x_train,51,2,deriv=2)
# x_test_2d=savgol_filter(x_test,51,2,deriv=2)

## Standardize on columns (Leaving out avocado data, the last item of the list)
x_train_scaled, x_test_scaled = standardize_column(x_train_1d, x_test_1d)


## plot the standardized test spectra of each individual fruit
plt.figure(figsize=(12,4))
plt.subplot(121)
plt.title('train')
plt.plot(w, x_test_scaled.T,  alpha=0.7)
plt.axhline(0,c='k')
plt.subplot(122)
plt.title('test')
plt.plot(w, x_test_scaled.T, alpha=0.7)
plt.axhline(0,c='k')
plt.tight_layout()
plt.show()


### CNN architectures

Lets create a simple CNN architecture composed of one convolutional layer with just 1 filter, one dense layer and a final output layer (for regression purposes). We will also add L2 regularization on all layers for helping stabilize the learning process and decrease overfitting problems. 

The adjustable hyperparameter will be the width of the convolutional filter, the number of units in the dense layer and the strength of the L2 regularization.

In [ ]:
## Run this function to make the computations reproducible
reproducible_comp()

## Define the model
def create_model_cnn(num_FC_units, filter_size,  reg_beta):
    ## Layers dimensions
    INPUT_DIMS = np.shape(x_train_scaled)[1]
    ## number of filters
    K_NUMBER = 1
    ## filter size
    K_WIDTH = filter_size
    ## filter stride / step
    K_STRIDE = 1
    ## number of units in the dense layer
    FC_UNITS = num_FC_units
    ## Output dimensions, 1 for regression
    REG_OUTPUT_DIMS = 1
    ## Global (all layers) L2 regularizer parameter
    K_REG = tf.keras.regularizers.l2(reg_beta)
    ## Weights initialization for multiple layers. Fixing a seed for reproducibility
    K_INIT = tf.keras.initializers.he_normal(seed=123)
    
    ####### Architecture of the main model ##############
    ## Input Layer
    input_layer = layers.Input(shape=(INPUT_DIMS,), name='INPUT')
    ## Reshape the input to 1D and to accommodate for batch size
    x = layers.Reshape((INPUT_DIMS, 1),name='RESHAPE')(input_layer)
    ## Convolutional layer
    x = layers.Conv1D(filters=K_NUMBER, 
                      kernel_size=K_WIDTH, 
                      strides=K_STRIDE, 
                      padding='same', 
                      kernel_initializer=K_INIT,
                      kernel_regularizer=K_REG,
                      activation='elu',
                      name='CONVOLUTIONAL')(x)
    ## Add flatten layer for reshaping the dimensions
    x = layers.Flatten(name='FLATTEN')(x)
    ## Fully connected layer
    x = layers.Dense(FC_UNITS, 
                     kernel_initializer=K_INIT, 
                     kernel_regularizer=K_REG,
                     activation='elu', 
                     name='DENSE')(x)
    ## Regression output
    reg_output = layers.Dense(REG_OUTPUT_DIMS, 
                              kernel_initializer=K_INIT, 
                              activation='linear', 
                              name='REG_OUTPUT')(x)

    # Create the model with multiple outputs
    model_cnn = Model(inputs=input_layer, outputs=[ reg_output], name='MODEL_CNN_V0')
    
    return model_cnn

Lets instantiate the model with some hyperparameters

In [ ]:
cnn_1 = create_model_cnn(92, 5, 0.03)

# Show the summary of the model
cnn_1.summary()

In [ ]:
## Plot the architecture
tf.keras.utils.plot_model(cnn_1, 
                          show_shapes=True, 
                          show_layer_activations=True, 
                          show_dtype=False , 
                          show_layer_names=False, 
                          rankdir='TB', 
                          expand_nested=False,  dpi=64)

The next step consists in training our CNN. For that purpose we define a few useful callback functions (functions that will run during training), compile our model choosing the type of gradient optimizer (ADAM in this case) and train the model.

We will use the same type of preprocessing that we used for the PLS model, i.e. 1st derivative.

We split the train set into calibration and validation subsets for the model training. The CNN will use the calibration data to train the model weights and the validation set for monitoring the process. This is useful to check for model overfitting. In this case will will use a callback named EarlyStopping that will also help on this.

In [ ]:
## Clear model parameter that might be in memory
keras.backend.clear_session()

########### Callbacks to use during training
## EarlyStopping: stop the training if the validation loss stops improving by "min_delta" over "patience" number of epochs
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', min_delta=1e-3, patience=52, mode='auto', restore_best_weights=True)

## ReduceLROnPlateau: Dynamicallyy reduces the learning rate by "factor" if the validation loss does not improve over "patience" epochs
rdlr = ReduceLROnPlateau(patience=25, factor=0.5, min_lr=1e-6, monitor='val_loss', verbose=0)

## This callback draws small progress bar in the screen for each training session. Its useful to check the progress of the task
from tqdm_progress_bar import TQDMProgressBar
progressbar = TQDMProgressBar(show_epoch_progress = False)
## Alternatively, we can monitor the training in real time using PlotLossesKerasTF (it is a bit slower)
liveplot = PlotLossesKerasTF()

## Save the best model based on the val loss (the val loss is not used at any point during training)
model_name = 'cnn1.keras'
checkpointer = ModelCheckpoint(filepath=model_name, monitor='loss', verbose=0, save_best_only=True)

################# Training data splitting
## Split train data into calibration and validation sets (even better if we use CV instead of a single split)
## First we split the train into calibration and validation sets.
x_train_cal, x_train_val, y_train_cal, y_train_val = train_test_split(x_train_1d, y_train, test_size=0.2, random_state=42)
## Then we use the calibration set statistics to standardize the calibration, validation and test sets
x_train_cal_scaled, x_test_scaled = standardize_column(x_train_cal, x_test_1d)
_, x_train_val_scaled = standardize_column(x_train_cal, x_train_val)

print('Train calibration set shape:', x_train_cal_scaled.shape)
print('Train validation set shape:', x_train_val_scaled.shape)
print('Test set shape:', x_test_scaled.shape)
#################


################## Define some of the training hyperparameters
## Number of samples in each batch 
BATCH_SIZE=256
## Learning rate (this is a heuristic value, it should be tuned)
LR=0.01*BATCH_SIZE/256.
print('Adam learning rate = {}'.format(LR))
## Number of epochs to train the model
EPOCHS=300

## Define the model hyperparameters
FC_UNITS = 92
FILTER_SIZE = 5
L2_REG = 0.03
## Create the model
cnn_1 = create_model_cnn(FC_UNITS, FILTER_SIZE, L2_REG)
## Compile the model, using the Adam optimizer, the loss function Mean Squared Error (MSE) because this is a regression problem
cnn_1.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR), loss="mse", metrics=["mse"])


## Train the model and visualize the training process
#### TRIAL 1 ########################################
# h1 = cnn_1.fit(x_train_cal_scaled, y_train_cal, batch_size = BATCH_SIZE, shuffle=False, epochs = EPOCHS,
#                    validation_data = (x_train_val_scaled, y_train_val) ,
#                    callbacks=[early_stop, rdlr, checkpointer, liveplot], verbose=0)

#### TRIAL 2 ########################################
## Alternatively, use progressbar to visualize the train. Pass the training into a history object "h1" for later use
h1 = cnn_1.fit(x_train_cal_scaled, y_train_cal, batch_size = BATCH_SIZE, shuffle=False, epochs = EPOCHS,
                   validation_data = (x_train_val_scaled, y_train_val) ,
                   callbacks=[early_stop, rdlr, checkpointer, progressbar], verbose=0)

print(f'\n Training completed... \n Loading best model weights from {model_name}...')

## After the model finishes the training, load the best model weights
cnn_1.load_weights(model_name)

! Try running the previous cell by commenting section "TRIAL 1" and uncomment "TRIAL 2".

Visualize the training history and compute error metrics on the calibration, validation and test sets.

In [ ]:
## Take a look at the training process by plotting the models history.
plt.figure(figsize=(6,3))
plt.plot(h1.history['loss'], label='Train loss')
plt.plot(h1.history['val_loss'], label='Val loss')
plt.yscale('log')
plt.ylabel('Loss')
plt.xlabel('Epochs')
# plt.ylim(0.5,1)
plt.legend()
## In case you used ReduceLROnPlateau() you can plot the lr as well
ax2 = plt.gca().twinx()
ax2.plot(h1.history['learning_rate'], color='r', ls='--')
ax2.set_ylabel('learning rate',color='r')
plt.tight_layout()
plt.show()


## Compute RMSE metrics for TRAIN and TEST sets
y_train_cal_pred = cnn_1.predict(x_train_cal_scaled)
y_train_val_pred = cnn_1.predict(x_train_val_scaled)
y_test_pred = cnn_1.predict(x_test_scaled)

## Compute train error scores
R2_train_cal = r2_score(y_train_cal, y_train_cal_pred)
rmse_train_cal = root_mean_squared_error(y_train_cal, y_train_cal_pred)
R2_train_val = r2_score(y_train_val, y_train_val_pred)
rmse_train_val = root_mean_squared_error(y_train_val, y_train_val_pred)

## Compute test error scores
R2_test = r2_score(y_test, y_test_pred)
rmse_test = root_mean_squared_error(y_test, y_test_pred)

print('\n\t ERROR METRICS: \t CALIB  \t\t VALID \t\t TEST')
print(f'\t R2: \t\t\t {R2_train_cal:.3f}  \t\t {R2_train_val:.3f} \t\t {R2_test:.3f}')
print(f'\t RMSE: \t\t\t {rmse_train_cal:.3f} \t\t\t {rmse_train_val:.3f} \t\t {rmse_test:.3f}' )

## Clear clutter from previous session
keras.backend.clear_session()
print('\n Keras backend cleared...')

In [ ]:
plot_prediction2(y_train_cal, y_test, y_train_cal_pred, y_test_pred, 'CNN 1', savefig=False, figname=None)

### Grid search for one hyperparameter...

In [ ]:
reproducible_comp()

## The data
x_train_cal, x_train_val, y_train_cal, y_train_val = train_test_split(x_train_1d, y_train, test_size=0.1, random_state=42)
## Then we use the calibration set statistics to standardize the calibration, validation and test sets
x_train_cal_scaled, x_test_scaled = standardize_column(x_train_cal, x_test_1d)
_, x_train_val_scaled = standardize_column(x_train_cal, x_train_val)


########### Callbacks to use during training
## EarlyStopping: stop the training if the validation loss stops improving by "min_delta" over "patience" number of epochs
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', min_delta=1e-3, patience=50, mode='auto', restore_best_weights=True)
## ReduceLROnPlateau: Dynamicallyy reduces the learning rate by "factor" if the validation loss does not improve over "patience" epochs
rdlr = ReduceLROnPlateau(patience=25, factor=0.5, min_lr=1e-6, monitor='val_loss', verbose=0)
## This callback draws small progress bar in the screen for each training session. Its useful to check the progress of the task
progressbar = TQDMProgressBar(show_epoch_progress = False)


################## Define some of the training hyperparameters
## Number of samples in each batch 
BATCH_SIZE=256
## Learning rate (this is a heuristic value, it should be tuned)
LR=0.01*BATCH_SIZE/256.
print('Adam learning rate = {}'.format(LR))
## Number of epochs to train the model
EPOCHS=300

## Hyperparameter ranges to explore
# n_filters = [5,15,25,35,45,55,61]
# n_units = [16,24,32,64,88,92,96, 128,256] 
l2_reguls = [0.1, 0.05, 0.02, 0.01, 0.005, 0.001, 0]

metrics = [] ## empty list to store the error metrics

################# Start of the training loop for the grid search
for l2_reg in l2_reguls:
    ## Clear model parameter that might be in memory
    keras.backend.clear_session()
    
    ## Save the best model based on the val loss (the val loss is not used at any point during training)
    model_name = 'cnn_1_l2='+str(l2_reg)+'.keras'
    checkpointer = ModelCheckpoint(filepath=model_name, monitor='loss', verbose=0, save_best_only=True)

    ## Define the model hyperparameters
    FC_UNITS = 92
    FILTER_SIZE = 5
    L2_REG = l2_reg ## <----- value of the search space
    ## Create the model. It is important to create a new model instance each time to make sure the weights are reset
    model = create_model_cnn(FC_UNITS, FILTER_SIZE, L2_REG)
    ## Compile the model, using the Adam optimizer, the loss function Mean Squared Error (MSE) because this is a regression problem
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR), loss="mse", metrics=["mse"])

    ## train the model 
    model.fit(x_train_cal_scaled, y_train_cal, batch_size = BATCH_SIZE, shuffle=False, epochs = 300,
              validation_data = (x_train_val_scaled, y_train_val) ,
              callbacks=[early_stop, rdlr, checkpointer, progressbar], verbose=0)

    print(f'\n Training completed... \n Loading best model weights from {model_name}...')
    ## Load the best model weights
    model.load_weights(model_name)    
    
    ## Compute RMSE metrics for TRAIN and TEST sets
    y_train_cal_pred = model.predict(x_train_cal_scaled)
    y_train_val_pred = model.predict(x_train_val_scaled)
    y_test_pred = model.predict(x_test_scaled)

    ## Compute train error scores
    R2_train_cal = r2_score(y_train_cal, y_train_cal_pred)
    rmse_train_cal = root_mean_squared_error(y_train_cal, y_train_cal_pred)
    R2_train_val = r2_score(y_train_val, y_train_val_pred)
    rmse_train_val = root_mean_squared_error(y_train_val, y_train_val_pred)

    ## Compute test error scores
    R2_test = r2_score(y_test, y_test_pred)
    rmse_test = root_mean_squared_error(y_test, y_test_pred)
    
    print('\n----------------------------')
    print(f'CNN with l2_reg = {l2_reg}')
    print('----------------------------')
    print('\n\t ERROR METRICS: \t CALIB  \t\t VALID \t\t TEST')
    print(f'\t R2: \t\t\t {R2_train_cal:.3f}  \t\t {R2_train_val:.3f} \t\t {R2_test:.3f}')
    print(f'\t RMSE: \t\t\t {rmse_train_cal:.3f} \t\t\t {rmse_train_val:.3f} \t\t {rmse_test:.3f}' )
    ## Append the metrics to the list
    metrics.append([l2_reg, rmse_train_cal, rmse_train_val, rmse_test, R2_train_cal, R2_train_val, R2_test])


    ## Clear clutter from previous session
    keras.backend.clear_session()
    print('\n Keras backend cleared...')
############### END OF GRID SEARCH LOOP ####################


## Convert the metrics list to a dataframe
metrics_df = pd.DataFrame(metrics, columns=['l2_reg', 'RMSE_train_cal', 'RMSE_train_val', 'RMSE_test', 'R2_train_cal', 'R2_train_val', 'R2_test'])

## Save the metrics to a csv file
# metrics_df.to_csv('cnn_1_metrics.csv', index=False)

Now that the grid search is over lets take a look at the results stored in our dataframe "metrics_df"

In [ ]:
metrics_df.round(3) ## rounding the values to 3 decimal places

By looking into the validation set metrics what is the best L2 regularization to use?

## 6) CNN analysis

**Visualize the filter of the conv. layer of model cnn_1**.

In [ ]:
## print the conv. filter weights of model cnn_1
for layer in cnn_1.layers:
    if 'CONVOLUTIONAL' in layer.name:
        print(f'Name: {layer.name}')
        print(f'Shape: {layer.get_weights()[0].shape}')
        print(f'Weights: {layer.get_weights()[0][:,:,0].flatten()}')
        print(f'Biases: {layer.get_weights()[1]}')

plt.figure(figsize=(4,2))
plt.plot(cnn_1.get_layer('CONVOLUTIONAL').get_weights()[0][:,:,0])
plt.ylabel('Filter weights')
plt.xlabel('Filter width')
plt.show()

**Visualize the activation of the conv. layer of model cnn_1**. This is done by creating a sub-model based on the initial model but that ends at the conv. layer.

In [ ]:
## define cnn_1 conv. output model. This model will output the output of the first convolutional layer of cnn_1
conv_out_1 = Model(inputs=cnn_1.input, outputs=cnn_1.get_layer('CONVOLUTIONAL').output)

## select a test sample to predict (original and spectra)
n = -1 ## last sample
test_sample = x_test_1d[n,:]
test_sample_scaled = x_test_scaled[n,:]


## use the conv_out_1 model to predict the samples. This gives us the output of the conv. layer
conv_out_1_pred = conv_out_1.predict(test_sample.reshape(-1, 490, 1)).reshape(-1,490)
conv_out_1_pred_scaled = conv_out_1.predict(test_sample_scaled.reshape(-1, 490, 1)).reshape(-1,490)


plt.figure(figsize=(12,3))
ax=plt.subplot(111)
plt.plot(w, test_sample, 'k', label='Original 1d spectra')
plt.plot(w, conv_out_1_pred[:,:].T - cnn_1.get_layer('CONVOLUTIONAL').get_weights()[1], 'b',label='Conv. layer output') ## subtract the bias
plt.legend(frameon=False, loc=1)
# plt.ylim(-5,2.5)
plt.xlabel('Wavelength (nm)')
plt.tight_layout()
# save png
# plt.savefig('filter_CNN1.png', dpi=200, bbox_inches='tight')
plt.show()


plt.figure(figsize=(12,3))
ax=plt.subplot(111)
plt.plot(w, test_sample_scaled, 'k', label='Scaled 1d spectra')
plt.plot(w, conv_out_1_pred_scaled[:,:].T - cnn_1.get_layer('CONVOLUTIONAL').get_weights()[1], 'b',label='Conv. layer output') ## subtract the bias
plt.legend(frameon=False, loc=4)
# plt.ylim(-5,2.5)
plt.xlabel('Wavelength (nm)')
plt.tight_layout()
# save png
# plt.savefig('filter_CNN1.png', dpi=200, bbox_inches='tight')
plt.show()

### SHAP values

Lets compute the SHAP values for our model cnn_1

In [ ]:
## Clear model parameter that might be in memory
keras.backend.clear_session()

########### Callbacks to use during training
## EarlyStopping: stop the training if the validation loss stops improving by "min_delta" over "patience" number of epochs
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', min_delta=1e-3, patience=52, mode='auto', restore_best_weights=True)

## ReduceLROnPlateau: Dynamicallyy reduces the learning rate by "factor" if the validation loss does not improve over "patience" epochs
rdlr = ReduceLROnPlateau(patience=25, factor=0.5, min_lr=1e-6, monitor='val_loss', verbose=0)

## This callback draws small progress bar in the screen for each training session. Its useful to check the progress of the task
from tqdm_progress_bar import TQDMProgressBar
progressbar = TQDMProgressBar(show_epoch_progress = False)
## Alternatively, we can monitor the training in real time using PlotLossesKerasTF (it is a bit slower)
liveplot = PlotLossesKerasTF()

## Save the best model based on the val loss (the val loss is not used at any point during training)
model_name = 'cnn1.keras'
checkpointer = ModelCheckpoint(filepath=model_name, monitor='loss', verbose=0, save_best_only=True)

################# Training data splitting
## Split train data into calibration and validation sets (even better if we use CV instead of a single split)
## First we split the train into calibration and validation sets.
x_train_cal, x_train_val, y_train_cal, y_train_val = train_test_split(x_train_1d, y_train, test_size=0.2, random_state=42)
## Then we use the calibration set statistics to standardize the calibration, validation and test sets
x_train_cal_scaled, x_test_scaled = standardize_column(x_train_cal, x_test_1d)
_, x_train_val_scaled = standardize_column(x_train_cal, x_train_val)

print('Train calibration set shape:', x_train_cal_scaled.shape)
print('Train validation set shape:', x_train_val_scaled.shape)
print('Test set shape:', x_test_scaled.shape)
#################


################## Define some of the training hyperparameters
## Number of samples in each batch 
BATCH_SIZE=256
## Learning rate (this is a heuristic value, it should be tuned)
LR=0.01*BATCH_SIZE/256.
print('Adam learning rate = {}'.format(LR))
## Number of epochs to train the model
EPOCHS=300

## Define the model hyperparameters
FC_UNITS = 92
FILTER_SIZE = 35
L2_REG = 0.02
## Create the model
cnn_1 = create_model_1(1, [FC_UNITS], FILTER_SIZE, [], L2_REG)
## Compile the model, using the Adam optimizer, the loss function Mean Squared Error (MSE) because this is a regression problem
cnn_1.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR), loss="mse", metrics=["mse"])


## Train the model and visualize the training process
#### TRIAL 1 ########################################
# h1 = cnn_1.fit(x_train_cal_scaled, y_train_cal, batch_size = BATCH_SIZE, shuffle=False, epochs = EPOCHS,
#                    validation_data = (x_train_val_scaled, y_train_val) ,
#                    callbacks=[early_stop, rdlr, checkpointer, liveplot], verbose=0)

#### TRIAL 2 ########################################
## Alternatively, use progressbar to visualize the train. Pass the training into a history object "h1" for later use
h1 = cnn_1.fit(x_train_cal_scaled, y_train_cal, batch_size = BATCH_SIZE, shuffle=False, epochs = EPOCHS,
                   validation_data = (x_train_val_scaled, y_train_val) ,
                   callbacks=[early_stop, rdlr, checkpointer, progressbar], verbose=0)

print(f'\n Training completed... \n Loading best model weights from {model_name}...')

## After the model finishes the training, load the best model weights
cnn_1.load_weights(model_name)

In [ ]:
# ## Compute RMSE metrics for TRAIN and TEST sets
# y_train_cal_pred = cnn_1.predict(x_train_cal_scaled)
# y_train_val_pred = cnn_1.predict(x_train_val_scaled)
# y_test_pred = cnn_1.predict(x_test_scaled)

# ## Compute train error scores
# R2_train_cal = r2_score(y_train_cal, y_train_cal_pred)
# rmse_train_cal = root_mean_squared_error(y_train_cal, y_train_cal_pred)
# R2_train_val = r2_score(y_train_val, y_train_val_pred)
# rmse_train_val = root_mean_squared_error(y_train_val, y_train_val_pred)

# ## Compute test error scores
# R2_test = r2_score(y_test, y_test_pred)
# rmse_test = root_mean_squared_error(y_test, y_test_pred)

# print('\n\t ERROR METRICS: \t CALIB  \t\t VALID \t\t TEST')
# print(f'\t R2: \t\t\t {R2_train_cal:.3f}  \t\t {R2_train_val:.3f} \t\t {R2_test:.3f}')
# print(f'\t RMSE: \t\t\t {rmse_train_cal:.3f} \t\t\t {rmse_train_val:.3f} \t\t {rmse_test:.3f}' )

# ## Clear clutter from previous session
# keras.backend.clear_session()
# print('\n Keras backend cleared...')

In [ ]:
import shap

###### Use the train data as basis/background for the SHAP values
shap_cnn_1 = shap.GradientExplainer(cnn_1, x_train_cal_scaled[:600,:])


###### Compute the shap values for the test data
shap_test_values_cnn_1 = shap_cnn_1.shap_values(x_test_scaled) 


In [ ]:

## Plot the mean abs(Shap) values
plt.figure(figsize=(12,4))
plt.title('Shap values of the CNN model')
plt.plot(w, np.mean(np.abs(shap_test_values_cnn_1), axis=0), 'r-', alpha=0.7, label='cnn_1')
plt.plot(w,5*np.mean(x_train_1d, axis=0), 'g-', lw=6,alpha=0.5, label='mean spectra')
plt.legend()
plt.ylabel('Mean abs(SHAP values)')
plt.grid(axis='x')
plt.tight_layout()
plt.show()

Compute the SHAP and VIP values of PLS model

In [ ]:
pls_model = PLSRegression(n_components=8, scale=True)
pls_model.fit(x_train_1d, y_train)
## compute PLS vip scores
pls_vip = vip(pls_model)

# ## compute the PLS shap values for comparison. We use 500 random sample from the train set to establish the background distribution
shap_train_pls = shap.Explainer(pls_model.predict, shuffle(x_train_1d, random_state=42))
shap_test_values_pls = shap_train_pls.shap_values(shuffle(x_test_1d, random_state=42))



In [ ]:
## Plot the mean abs(Shap) values
plt.figure(figsize=(12,4))
plt.title('Shap values of the CNN model')
plt.plot(w, np.mean(np.abs(shap_test_values_pls), axis=0), 'r-', alpha=0.7, label='cnn_1')
plt.plot(w,5*np.mean(x_train_1d, axis=0), 'g-', lw=6,alpha=0.5, label='mean spectra')
plt.legend()
plt.ylabel('Mean abs(SHAP values)')
plt.grid(axis='x')
plt.tight_layout()
plt.show()

### 6.4) Regression coefficients

In this case we implment the regression coefficients as introduced by Cui, Fearn 2018.

In [ ]:
## Compute regression coeffs according to eq. (7) of the paper
def compute_regression_coeffs(x_subset, model, epsilon):
    ## create a prediction for the non-perturbed spectra
    y_pred = model.predict(x_subset, verbose=0)
    
    ## Create a matrix to store the regression coefficients
    w_reg = np.zeros((x_subset.shape[0],x_subset.shape[1]))

    ## For each feature i compute the regression coefficients
    for i in np.arange(x_subset.shape[1]):
        ## create array with wavelenghts (aka features)
        x_pert=x_subset.copy()
        ## apply epsilon pertubation to feature i only
        x_pert[:,i]=x_subset[:,i]+epsilon
        ## compute new model prediction where the input is the locally perturbed spectra
        y_pred_pert=model.predict(x_pert, verbose=0)
        ## compute the regression coefficients by subtracting the non-perturbed prediction from the perturbed prediction
        w_reg[:,i] = np.divide((y_pred_pert[:,0] - y_pred[:,0]),epsilon)
        ## normalize the regression coefficients by dividing by the maximum value
        #w_reg[:,i] = w_reg[:,i]/np.max(np.abs(w_reg[:,i]))
    return w_reg


## exactly the same function as before but without the verbose flag (not needed to certain models, like PLS)
def compute_regression_coeffs_mod(x_subset, model, epsilon):
    ## create a prediction for the non-perturbed spectra
    y_pred = model.predict(x_subset)
    
    ## Create a matrix to store the regression coefficients
    w_reg = np.zeros((x_subset.shape[0],x_subset.shape[1]))

    ## For each feature i compute the regression coefficients
    for i in np.arange(x_subset.shape[1]):
        ## create array with wavelenghts (aka features)
        x_pert=x_subset.copy()
        ## apply epsilon pertubation to feature i only
        x_pert[:,i]=x_subset[:,i]+epsilon
        ## compute new model prediction where the input is the locally perturbed spectra
        y_pred_pert=model.predict(x_pert)
        ## compute the regression coefficients by subtracting the non-perturbed prediction from the perturbed prediction
        w_reg[:,i] = np.divide((y_pred_pert[:,0] - y_pred[:,0]),epsilon)
        ## normalize the regression coefficients by dividing by the maximum value
        #w_reg[:,i] = w_reg[:,i]/np.max(np.abs(w_reg[:,i]))
    return w_reg


## This is the same function as before but we used it for models that due dual output
## and in this case the regression prediction
def compute_regression_coeffs_2_reg(x_subset, model, epsilon):
    ## create a prediction for the non-perturbed spectra
    y_pred = model.predict(x_subset, verbose=0)[1] ## regression is the second output, hence [1]
    
    ## Create a matrix to store the regression coefficients
    w_reg = np.zeros((x_subset.shape[0],x_subset.shape[1]))

    ## For each feature i compute the regression coefficients
    for i in np.arange(x_subset.shape[1]):
        ## create array with wavelenghts (aka features)
        x_pert=x_subset.copy()
        ## apply epsilon pertubation to feature i only
        x_pert[:,i]=x_subset[:,i]+epsilon
        ## compute new model prediction where the input is the locally perturbed spectra
        y_pred_pert=model.predict(x_pert, verbose=0)[1] ## regression is the second output, hence [1]
        ## compute the regression coefficients by subtracting the non-perturbed prediction from the perturbed prediction
        w_reg[:,i] = np.divide((y_pred_pert[:,0] - y_pred[:,0]),epsilon)
        ## normalize the regression coefficients by dividing by the maximum value
        #w_reg[:,i] = w_reg[:,i]/np.max(np.abs(w_reg[:,i]))
    return w_reg

## This is the same function as before but we used it for models that due dual output
## and in this case the classification prediction
def compute_regression_coeffs_2_class(x_subset, model, epsilon):
    ## create a prediction for the non-perturbed spectra
    y_pred = model.predict(x_subset, verbose=0)[0] ## classifictio is the first output, hence [0]
    
    ## Create a matrix to store the regression coefficients
    w_reg = np.zeros((x_subset.shape[0],x_subset.shape[1]))

    ## For each feature i compute the regression coefficients
    for i in np.arange(x_subset.shape[1]):
        ## create array with wavelenghts (aka features)
        x_pert=x_subset.copy()
        ## apply epsilon pertubation to feature i only
        x_pert[:,i]=x_subset[:,i]+epsilon
        ## compute new model prediction where the input is the locally perturbed spectra
        y_pred_pert=model.predict(x_pert, verbose=0)[0] ## classifictio is the first output, hence [0]
        ## compute the regression coefficients by subtracting the non-perturbed prediction from the perturbed prediction
        w_reg[:,i] = np.divide((y_pred_pert[:,0] - y_pred[:,0]),epsilon)
        ## normalize the regression coefficients by dividing by the maximum value
        #w_reg[:,i] = w_reg[:,i]/np.max(np.abs(w_reg[:,i]))
    return w_reg


In [ ]:
## Select the a subset for computing the regression coefficients. In this case the whole test set
x_subset = x_test_scaled
## Amplitude of the perturbation
epsilon = 5e-5
## Compute the regression coefficients for the subset
w_cnn_1 = compute_regression_coeffs(x_subset, cnn_1, epsilon )


# plot the regression coefficients
plt.figure(figsize=(12,3))
plt.plot(w, np.mean(np.abs(w_cnn_1), axis=0))
# plt.plot(w[wi:wf], np.abs(w_pls.T), 'k')
plt.xlabel('Wavelength')
plt.ylabel('Regression Coefficients')
plt.title('Regression Coefficients for CNN_1')
plt.show()
